# 🚀 What this model should  learn

```
Input: John bought 3 apples for $5
Output:
{
  "name": "John",
  "item": "apples",
  "quantity": 3,
  "price": 5
}
```


In [ ]:
# 🧾 1. Install

# !pip install transformers datasets


In [24]:
# 📚 2. Create Dataset (VERY IMPORTANT)

# 👉 Use **multiple patterns** (not just one)

from datasets import Dataset

data = [
    {"input": "John bought 3 apples for $5",
     "output": '{"name": "John", "item": "apples", "quantity": 3, "price": 5}'},

    {"input": "Alice purchased 2 books for $20",
     "output": '{"name": "Alice", "item": "books", "quantity": 2, "price": 20}'},

    {"input": "Bob bought 5 oranges for $10",
     "output": '{"name": "Bob", "item": "oranges", "quantity": 5, "price": 10}'},

    {"input": "Emma purchased 1 laptop for $1000",
     "output": '{"name": "Emma", "item": "laptop", "quantity": 1, "price": 1000}'},

    {"input": "Noah bought 4 pens for $8",
     "output": '{"name": "Noah", "item": "pens", "quantity": 4, "price": 8}'},

     {"input": "John bought 3 apples for $5",
     "output": '{"name": "John", "item": "apples", "quantity": 3, "price": 5}'},

    {"input": "Alice purchased 2 books for $20",
     "output": '{"name": "Alice", "item": "books", "quantity": 2, "price": 20}'},

    {"input": "Bob bought 5 oranges for $10",
     "output": '{"name": "Bob", "item": "oranges", "quantity": 5, "price": 10}'},

    {"input": "Emma purchased 1 laptop for $1000",
     "output": '{"name": "Emma", "item": "laptop", "quantity": 1, "price": 1000}'},

    {"input": "Noah bought 4 pens for $8",
     "output": '{"name": "Noah", "item": "pens", "quantity": 4, "price": 8}'},

    {"input": "Liam purchased 7 notebooks for $35",
     "output": '{"name": "Liam", "item": "notebooks", "quantity": 7, "price": 35}'},

    {"input": "Olivia bought 6 bananas for $12",
     "output": '{"name": "Olivia", "item": "bananas", "quantity": 6, "price": 12}'},

    {"input": "Ava purchased 3 chairs for $150",
     "output": '{"name": "Ava", "item": "chairs", "quantity": 3, "price": 150}'},

    {"input": "William bought 2 tables for $300",
     "output": '{"name": "William", "item": "tables", "quantity": 2, "price": 300}'},

    {"input": "Sophia purchased 5 bottles for $25",
     "output": '{"name": "Sophia", "item": "bottles", "quantity": 5, "price": 25}'},

    {"input": "James bought 8 markers for $16",
     "output": '{"name": "James", "item": "markers", "quantity": 8, "price": 16}'},

    {"input": "Isabella purchased 2 bags for $80",
     "output": '{"name": "Isabella", "item": "bags", "quantity": 2, "price": 80}'},

    {"input": "Benjamin bought 10 pencils for $10",
     "output": '{"name": "Benjamin", "item": "pencils", "quantity": 10, "price": 10}'},

    {"input": "Mia purchased 1 phone for $700",
     "output": '{"name": "Mia", "item": "phone", "quantity": 1, "price": 700}'},

    {"input": "Lucas bought 9 apples for $18",
     "output": '{"name": "Lucas", "item": "apples", "quantity": 9, "price": 18}'},

    {"input": "Charlotte purchased 4 dresses for $200",
     "output": '{"name": "Charlotte", "item": "dresses", "quantity": 4, "price": 200}'},

    {"input": "Henry bought 6 cups for $24",
     "output": '{"name": "Henry", "item": "cups", "quantity": 6, "price": 24}'},

    {"input": "Amelia purchased 3 shoes for $150",
     "output": '{"name": "Amelia", "item": "shoes", "quantity": 3, "price": 150}'},

    {"input": "Alexander bought 2 laptops for $2000",
     "output": '{"name": "Alexander", "item": "laptops", "quantity": 2, "price": 2000}'},

    {"input": "Evelyn purchased 5 toys for $50",
     "output": '{"name": "Evelyn", "item": "toys", "quantity": 5, "price": 50}'},


]

dataset = Dataset.from_list(data)


In [25]:
# 🧠 3. Format for Training

# 👉 This step is CRITICAL

def format_example(example):
    return {
        "input": f"Extract structured data as JSON:\n{example['input']}",
        "output": example["output"]
    }


dataset = dataset.map(format_example)


Map:   0%|          | 0/25 [00:00<?, ? examples/s]

In [26]:
# 🤖 4. Load Model

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)


Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [28]:
def generate(model, text):
    prompt = f"Extract structured data as JSON:\n{text}"

    inputs = tokenizer(prompt, return_tensors="pt")

    outputs = model.generate(
        **inputs,
        max_new_tokens=50,
        repetition_penalty=1.5,
        no_repeat_ngram_size=2
    )

    result = tokenizer.decode(outputs[0], skip_special_tokens=True)


    return result

print("Before Fine-Tuning:")
print(generate(model, "Michael bought 6 bananas for $12"))


Before Fine-Tuning:
Michael bought 6 bananas for $12


In [29]:
# 🔢 5. Tokenization (IMPORTANT FIX INCLUDED)

def tokenize(example):
    inputs = tokenizer(
        example["input"],
        truncation=True,
        padding="max_length",
        max_length=64
    )

    targets = tokenizer(
        example["output"],
        truncation=True,
        padding="max_length",
        max_length=64
    )

    inputs["labels"] = targets["input_ids"]
    return inputs

tokenized_dataset = dataset.map(tokenize, batched=True)


Map:   0%|          | 0/25 [00:00<?, ? examples/s]

In [30]:
# 🏋️ 6. Train

from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./model",
    per_device_train_batch_size=2,
    num_train_epochs=20,
    logging_steps=1,
    learning_rate=5e-5
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset
)

trainer.train()


Step,Training Loss
1,10.819355
2,10.749221
3,10.624520
4,10.683279
5,10.615923
6,10.608364
7,10.543759
8,10.504785
9,10.543696
10,10.497802


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=260, training_loss=8.730136069884667, metrics={'train_runtime': 479.2781, 'train_samples_per_second': 1.043, 'train_steps_per_second': 0.542, 'total_flos': 11618156544000.0, 'train_loss': 8.730136069884667, 'epoch': 20.0})

In [31]:
# 🔮 7. Inference (VERY IMPORTANT)

def generate(text):
    prompt = f"Extract structured data as JSON:\n{text}"

    inputs = tokenizer(prompt, return_tensors="pt")

    outputs = model.generate(
    **inputs,
    max_new_tokens=50,
    repetition_penalty=1.5,
    no_repeat_ngram_size=2
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

print(generate("Michael bought 6 bananas for $12"))




6:6, 12


In [ ]:

# 🎯 Expected Output

```
{"name": "Michael", "item": "bananas", "quantity": 6, "price": 12}
```


In [18]:
print(dataset[0])

{'input': 'John bought 3 apples for $5', 'output': '{"name": "John", "item": "apples", "quantity": 3, "price": 5}', 'text': '### Instruction:\nExtract structured data\n\n### Input:\nJohn bought 3 apples for $5\n\n### Output:\n{"name": "John", "item": "apples", "quantity": 3, "price": 5}'}


In [ ]:



* LLMs can do **information extraction**
* Fine-tuning = enforcing structure

---

###  Key point

> “We are not teaching the model language—we are teaching it structure.”

---

# ⚠️ If output is not perfect

That’s NORMAL. Fix by:

* Adding more examples (10–20 total)
* Keeping format consistent
* Lowering temperature (0.2–0.3)

---

# 🚀 If you want next level

I can upgrade this into:

* ✅ JSON validation (auto-check correctness)
* ✅ UI demo (students LOVE this)
* ✅ Compare with prompting vs fine-tuning

Just tell me 👍
